In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [4]:
# Datasets and Dataloaders
from torch.utils.data import DataLoader            
import torchvision.transforms as transforms        # transform is a torchvision utility which helps to perform transformations on image    

# image => scale (0,1) => normalize (-1,1)
transform=transforms.Compose([                    # it helps to chain lot of image transformations( multiple transformations on image like scale,normalize..), we are now setting the transformation which will be applied on each image later
    transforms.ToTensor(),                           # it converts all images into pytorch tensors + automatically scale all images
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))          # it normalize all images and after normalization define the desired standard deviation and mean inside ()
])
trainset = CIFAR10(
 root="./data",
 train=True,
 download=False,
 transform=transform
)
testset = CIFAR10(
 root="./data",
 train=False,
 download=False,
 transform=transform
)
 


In [5]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [6]:
trainLoader=DataLoader(trainset,batch_size=64,shuffle=True)
testLoader=DataLoader(testset,batch_size=64)

# Build the CNN

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(

            # 1st layer
            nn.Conv2d(3, 32 ,kernel_size=3,padding=1),      # ( no.of input channels, no.of output channels, kernel_size, padding )     
            nn.ReLU(),
            nn.MaxPool2d(2,2),                              # kernel_size=2, stride=2

            # 2nd layer
            nn.Conv2d(32, 64 ,kernel_size=3,padding=1),      # ( no.of input channels=no. of output channels of last layer, double of no.of output channels of last layer, kernel_size, padding )     
            nn.ReLU(),
            nn.MaxPool2d(2,2),                              # kernel_size=2, stride=2

            # 3rd layer
            nn.Conv2d(64, 128 ,kernel_size=3,padding=1),      # ( no.of input channels=no. of output channels of last layer, double of no.of output channels of last layer, kernel_size, padding )     
            nn.ReLU(),
            nn.MaxPool2d(2,2),                              # kernel_size=2, stride=2
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128, 256),                        # ( input values, output neurons(random) )
            nn.ReLU(),

            nn.Linear(256,10)                              # ( output values, output classes needed )
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)                          # flattening
        x=self.fc_layers(x)

        return x

In [8]:
model=CNN()

In [9]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

# Training the CNN

In [10]:
epochs=10

for epoch in range(epochs):
    epoch_training_loss=0.0

    for images,labels in trainLoader:
        optimizer.zero_grad()

        output=model.forward(images) # FP
        loss=criterion(output,labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss+=loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainLoader)}")

epoch = 1/10 & loss = 1.3748808507724186
epoch = 2/10 & loss = 0.9395819746929667
epoch = 3/10 & loss = 0.7513193012122303
epoch = 4/10 & loss = 0.6211686794029172
epoch = 5/10 & loss = 0.513865452707576
epoch = 6/10 & loss = 0.40892001552045193
epoch = 7/10 & loss = 0.32570767410271
epoch = 8/10 & loss = 0.25134292006244896
epoch = 9/10 & loss = 0.1931876177182588
epoch = 10/10 & loss = 0.15264380763730276


In [11]:
# Evaluate our model

correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images,labels in testLoader:
        outputs=model.forward(images)
        _,predicted=torch.max(outputs,1)

        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)
print(f"accuracy={correct_labels/total_labels * 100}")

accuracy=74.98
